In [1]:
# 导入必要的库
import numpy as np                          # 数值计算库
from vpsto.vpsto import VPSTO, VPSTOOptions # VPSTO轨迹优化库
from vpsto.obf import OBF                   # 椭球基函数库
import matplotlib.pyplot as plt             # 绘图库
import matplotlib.patches as patches        # matplotlib几何图形补丁

# Jupyter notebook魔法命令，用于自动重新加载模块
%load_ext autoreload
%autoreload 2

In [ ]:
# 定义2D 3自由度 机械臂类，包含正向运动学计算
class Manipulator():
    def __init__(self):
        # 机械臂链接长度：长度均为1
        self.l = np.array([1, 1, 1]) # link lengths
        # 关节角度限制：第一关节[0, π]，第二关节[-π, 0]
        self.q_min = np.array([0., -np.pi, -np.pi])    # 关节角度下限
        self.q_max = np.array([np.pi, 0., np.pi])     # 关节角度上限

    # 定义机械臂的正向运动学
    def fk(self, q):
        """
        正向运动学函数：根据关节角度计算机械臂各关节的笛卡尔坐标位置
        参数：
        q: 2x1数组，包含所有关节的角度
        返回：(N+1，2)矩阵，每行都是一个二维坐标位置，包含基座、所有关节和末端执行器的二维坐标位置
        """
        x0 = np.zeros(2)  # 基座位置（原点）
        # 第一关节位置：基座 + 第一段链接的笛卡尔位置
        x1 = x0 + self.l[0] * np.array([np.cos(q[0]), np.sin(q[0])])
        # ：第一关节 + 第二段链接的笛卡尔位置
        x2 = x1 + self.l[1] * np.array([np.cos(q[0] + q[1]), np.sin(q[0] + q[1])])
        # 末端执行器位置: 第二关节 + 第三段链接的笛卡尔位置
        x3 = x2 + self.l[2] * np.array([np.cos(q[0] + q[1] + q[2]), np.sin(q[0] + q[1] + q[2])])
        # 垂直堆栈列向量
        return np.vstack((x0, x1, x2, x3))  # 返回所有关节的位置

# 创建机械臂实例并测试正向运动学
# mp = Manipulator()  # 创建机械臂实例
# q = np.array([np.pi/4, -np.pi/2, np.pi/4])  # 定义关节角度
# positions = mp.fk(q)  # 计算正向运动学，获取各关节位置
# print("关节位置：\n", positions)  # 打印关节位置
# print("pos维度：", positions.shape)  # 打印位置数组的维度


In [1]:
'''
Author: Fang Kai[thissfk@qq.com]
Date: 2025-09
LastEditors: Fang Kai[thissfk@qq.com]
LastEditTime: 2025-09
FilePath: 2D_3dof_arm_av_collision.ipynb
Description: 
           If you need more information,
please contact Fang Kai[thissfk@qq.com] to get an access.   
Copyright (c) 2025 by Fang Kai, All Rights Reserved. 
'''
is_tested = 0
# 碰撞环境类：包含球形障碍物和工作空间边界
class CollisionEnvironment():
    def __init__(self):
        # 球形障碍物的中心位置
        # self.x = np.array([0.5, 0.5])
        # !!!test
        self.x = np.array([1, 1.5])
        # 球形障碍物的半径
        self.r = 0.1
        # 球形障碍物半径的平方（用于优化碰撞检测计算）
        self.r_sq = self.r**2

        # 工作空间的边界限制
        self.x_min = np.array([-0.75, 0.])     # 工作空间最小边界
        self.x_max = np.array([1.5, 1.5])     # 工作空间最大边界

    def isCollision(self, pts):
        """
        检查线段是否与球形障碍物发生碰撞
        参数：
        pts: (n, 4)数组，每行包含一条线段的两个二维端点坐标
        返回：布尔数组，表示每条线段是否与障碍物碰撞
        """
        # 计算线段方向向量
        e12 = pts[:,2:] - pts[:,:2]
        # 从线段起点到障碍物中心的向量
        e1x = self.x - pts[:,:2]
        
        """ test """
        if is_tested:
            print("\n e12 shape: \n",e12.shape)
            print("\n e12:\n",e12)
            print("\n e1x:\n",e1x)
            print("\n e12*e1x shape :",(e12 * e1x).shape)

        # 计算线段上距离障碍物中心最近点的参数λ（clip函数限制大小在0-1之间）
        e12_dot_e1x = np.sum(e12 * e1x, axis=1) # 压缩了列维度
        e12_dot_e12 = np.sum(e12**2, axis=1) # 压缩了列维度
        lam = np.clip(e12_dot_e1x / e12_dot_e12, 0, 1)

        """ test """
        if is_tested:
            print("\n e12_dot_e1x shape: \n",e12_dot_e1x.shape )
            print("\n e12_dot_e1x: \n",e12_dot_e1x )
            print("\n e12_dot_e12 shape: \n",e12_dot_e12.shape )
            print("\n e12_dot_e12: \n",e12_dot_e12 )
            print("\n lam: \n",lam )
            print("\n lam shape: \n",lam.shape )
            e12T = e12.T  # 转置以便广播运算
            lam_new_axis = lam[:, np.newaxis]  # 增加新轴以便广播运算
            print("\n lam_new_axis: \n",lam_new_axis)
            print("\n lam_new_axis shape: \n",lam_new_axis.shape )
            print("\n e12T: \n",e12T)
            print("\n lam_new_axis * e12: \n",lam_new_axis * e12)
            print("\n lam*e12T: \n",lam*e12T)
            print("\n (lam*e12T).T : \n",(lam*e12T).T)


        # 计算线段上最近点到障碍物中心的距离平方
        lam = lam[:,np.newaxis]
        d_sq = np.sum((e1x - lam * e12)**2, axis=1)
        # 如果距离小于障碍物半径，则发生碰撞
        return d_sq < self.r_sq
    
    def isRobotCollision(self, pts):
        """
        检查机械臂当前配置是否与障碍物发生碰撞
        参数：
        pts: (n, 2)数组，机械臂运动学链上每个点的二维坐标
        返回：布尔值，表示是否发生碰撞
        """
        # 将机械臂关节连接线段格式化
        pts_ = np.empty((pts.shape[0]-1, 4)) # 创建一个形状为 (n-1, 4) 的空数组
        # np的切片操作，区间左闭右开
        pts_[:,:2] = pts[:-1]  # 线段起点
        pts_[:,2:] = pts[1:]   # 线段终点
        # np.any 用于判断数组中是否存在至少一个元素为 True。如果存在，则返回 True，否则返回 False
        return np.any(self.isCollision(pts_))
        
    def isTrajectoryCollision(self, pts):
        """
        检查轨迹是否与障碍物发生碰撞
        参数：
        pts: (k, n, 2)数组，k是时间步数，n是机械臂关节数，2是二维坐标
             每个矩阵代表机械臂的运动学链，每行是机械臂链上的一个二维点
        返回：长度为k的布尔数组，每个元素表示该时间步是否发生碰撞
        """
        # 将机械臂链段重构为线段格式进行碰撞检测
        pts_ = np.empty((pts.shape[0] * (pts.shape[1]-1), 4))
        pts_[:,:2] = pts[:,:-1].reshape(-1, 2)  # 线段起点
        pts_[:,2:] = pts[:,1:].reshape(-1, 2)   # 线段终点
        # 检测每个时间步的碰撞情况
        collisions_over_time = np.any(self.isCollision(pts_).reshape(pts.shape[0], pts.shape[1]-1), axis=1)
        return collisions_over_time
    

# 创建碰撞环境实例并测试碰撞检测
# env = CollisionEnvironment()  # 创建碰撞环境实例
# mp = Manipulator()  # 创建机械臂实例
# q = np.array([np.pi/4, 0., 0.])  # 定义关节角度
# positions = mp.fk(q)  # 计算正向运动学，获取各关节位置
# print("\n 关节坐标:\n", positions)  # 打印关节位置

# # 画出关节位置和障碍物
# fig, ax = plt.subplots()
# ax.plot(positions[:,0], positions[:,1], '-o', label='arm pos')
# # 画出障碍物
# circle = patches.Circle(env.x, env.r, color='r', alpha=0.5, label='obstacle')
# ax.add_patch(circle)
# # 设置工作空间边界
# ax.set_aspect('equal', 'box')
# ax.grid()
# ax.legend()
# plt.title('pos')
# plt.show()

# env.isRobotCollision(positions)